# Re-export and publish a monolithic ONNX for `Stffens/bge-small-rrf-v2`

The v5 notebook exported `model.onnx` using `torch.onnx.export` and
the resulting file is a 1.26 MB stub that references an external
`model.onnx.data` file, which was never uploaded to the HF repo.
ONNX Runtime crashes at session init as a result:

```
filesystem error: cannot get file size: No such file or directory
[.../model.onnx.data]
```

BGE-small is 33M params -> ~130 MB as FP32 ONNX, well under the 2 GB
protobuf limit. Re-export as a single monolithic file using Optimum
(which exposes `no_external_data` explicitly), verify it loads with
onnxruntime, and upload back to the repo.

Runtime on a T4: ~2-3 min total. No GPU required for export (Optimum
handles it on CPU).

**You need an HF write token**: Put it in Colab Secrets as `HF_TOKEN`
with write access to `Stffens/bge-small-rrf-v2`.

In [ ]:
# Cell 1: Install Optimum + dependencies.
import os

os.chdir("/")
os.chdir("/content")

!pip install -q 'optimum[exporters]>=1.20' 'onnxruntime>=1.18' 'sentence-transformers>=3' huggingface_hub

In [ ]:
# Cell 2: Authenticate with HuggingFace using the token stored in Colab Secrets.
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import whoami

print(whoami())

In [ ]:
# Cell 3: Export to monolithic ONNX via Optimum.
# The key flag is no_post_process / use_external_data_format=False.
# main_export pulls the SentenceTransformer model from HF (safetensors,
# which the repo has) and produces a single-file ONNX locally.
import shutil
from pathlib import Path

from optimum.exporters.onnx import main_export

MODEL_ID = "Stffens/bge-small-rrf-v2"
OUT = Path("/content/rrf_v2_onnx")

if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

main_export(
    model_name_or_path=MODEL_ID,
    output=str(OUT),
    task="feature-extraction",
    opset=14,
    use_external_data_format=False,  # <-- the critical flag
    optimize="O1",
)

for p in sorted(OUT.iterdir()):
    size = p.stat().st_size
    print(f"  {p.name:40s}  {size / 1024 / 1024:7.2f} MB")

In [ ]:
# Cell 4: Verify the exported model loads and produces sane embeddings.
# Load with onnxruntime and compare against the safetensors ground truth
# from sentence-transformers. Cosine similarity should be > 0.999.
import numpy as np
import onnxruntime as ort
from sentence_transformers import SentenceTransformer
from tokenizers import Tokenizer

onnx_path = OUT / "model.onnx"
tok_path = OUT / "tokenizer.json"

sess = ort.InferenceSession(str(onnx_path))
print(f"ONNX session init OK. File size: {onnx_path.stat().st_size / 1024 / 1024:.2f} MB")

tokenizer = Tokenizer.from_file(str(tok_path))
tokenizer.enable_truncation(max_length=512)


def embed_onnx(text):
    enc = tokenizer.encode(text)
    ids = np.array([enc.ids], dtype=np.int64)
    mask = np.array([enc.attention_mask], dtype=np.int64)
    feed = {"input_ids": ids, "attention_mask": mask}
    if "token_type_ids" in {i.name for i in sess.get_inputs()}:
        feed["token_type_ids"] = np.zeros_like(ids)
    out = sess.run(None, feed)[0]  # last_hidden_state
    mask_exp = mask[:, :, None].astype(np.float32)
    v = (out * mask_exp).sum(axis=1) / mask_exp.sum(axis=1).clip(min=1e-9)
    v = v / np.linalg.norm(v, axis=1, keepdims=True).clip(min=1e-9)
    return v[0]


st = SentenceTransformer(MODEL_ID)

probes = [
    "What is reciprocal rank fusion?",
    "Vitamin C cures the common cold.",
    "Attention is all you need.",
]
for text in probes:
    v_onnx = embed_onnx(text)
    v_st = st.encode(text, normalize_embeddings=True)
    cos = float(v_onnx @ v_st)
    print(f"  cos={cos:.6f}  ({text[:40]})")

In [ ]:
# Cell 5: Upload the fixed model.onnx to the HF repo.
# This replaces the broken 1.26 MB stub with a ~130 MB monolithic file.
# Only touches model.onnx; tokenizer.json and other files are already
# correct in the repo.
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj=str(onnx_path),
    path_in_repo="model.onnx",
    repo_id=MODEL_ID,
    commit_message="Re-export model.onnx as monolithic (no external data)",
)
print(f"Uploaded {onnx_path} -> {MODEL_ID}/model.onnx")

In [ ]:
# Cell 6: End-to-end smoke: clear local HF cache for this model and
# re-download to confirm a fresh pull works with ONNX Runtime.
import shutil
from pathlib import Path

cache = Path.home() / ".cache/huggingface/hub"
for d in cache.glob("models--Stffens--bge-small-rrf-v2*"):
    shutil.rmtree(d, ignore_errors=True)
    print("cleared", d)

from huggingface_hub import hf_hub_download

p = hf_hub_download(MODEL_ID, "model.onnx")
print(f"downloaded: {p}  ({Path(p).stat().st_size / 1024 / 1024:.2f} MB)")

import onnxruntime as ort

sess = ort.InferenceSession(p)
print("Fresh pull -> ort.InferenceSession init OK")